In [27]:
import sagemaker
import boto3
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from io import StringIO
#from sagemaker.core.helper.session_helper import Session
from sagemaker.session import Session
from sagemaker.session import get_execution_role
import os

testing_mode = True
balanced = True

#Good AWS way of doing it, but costs $.40 a month per secret...
# # Retrieve role ARN from Parameter Store
# ssm_client = boto3.client('ssm')
# response = ssm_client.get_parameter(
#     Name='/sagemaker/execution-role-arn',
#     WithDecryption=True
# )
# role_arn = response['Parameter']['Value']

# Use with SageMaker
#sagemaker_session = Session()




# sess = Session()
# bucket = sess.default_bucket()
# #role = get_execution_role()
# region = boto3.Session().region_name

session = boto3.Session(region_name='us-east-1', profile_name='sagemaker')
sm = Session(boto_session=session)
s3 = session.client('s3')
#sm = Session()
#s3 = sm.boto_session.client('s3')
#region = sm.boto_region_name
#role_arn = os.environ.get('SAGEMAKER_EXECUTION_ROLE_ARN')
role = get_execution_role(sagemaker_session=sm)
print(f"Using role: {role}")



bucket_name = 'jrm-kaggle'
prefix = 'playgrounds6ep4/'
object_name = 'train.csv'
csv_obj = s3.get_object(Bucket=bucket_name, Key=prefix+object_name)
body = csv_obj['Body']
csv_string = body.read().decode('utf-8')

df_tv = pd.read_csv(StringIO(csv_string))
#print(df_tv.head())

df_x = df_tv.iloc[:,1:-1]

df_dummy = pd.get_dummies(df_x, dtype=int, drop_first=False)
continous_variables = df_dummy.select_dtypes(['float64']).columns
index = [df_dummy.columns.get_loc(col) for col in continous_variables]

x = df_dummy.iloc[:,:].values
y = df_tv.iloc[:,-1].values

class_le = LabelEncoder()
y = class_le.fit_transform(y)

if testing_mode:
    from sklearn.model_selection import train_test_split

    X_train, X_test, y_train, y_test = \
        train_test_split(x, y, 
                        test_size=0.20,
                        stratify=y,
                        random_state=1)
else:
    X_train, y_train = x, y

sc = StandardScaler().fit(X_train[:, index])
X_train[:, index] = sc.transform(X_train[:, index])

if testing_mode:
    X_test[:, index] = sc.transform(X_test[:, index])


if balanced:
    majority_class = np.argmax(np.bincount(y_train))
    minority_class = np.argmin(np.bincount(y_train))
    middle_class = list(set(np.unique(y_train)) - set([majority_class, minority_class]))[0]
    X_train_majority = X_train[y_train == majority_class]
    y_train_majority = y_train[y_train == majority_class]
    
    X_train_minority = X_train[y_train == minority_class]
    y_train_minority = y_train[y_train == minority_class]
    
    X_train_middle = X_train[y_train == middle_class]
    y_train_middle = y_train[y_train == middle_class]
    
    
    X_train_minority_upsampled, y_train_minority_upsampled = resample(X_train_minority, y_train_minority,
                                                                      replace=True,
                                                                      n_samples=X_train_middle.shape[0],
                                                                      random_state=1)
    X_train_majority_downsampled, y_train_majority_downsampled = resample(X_train_majority, y_train_majority,
                                                                      replace=False,
                                                                      n_samples=X_train_middle.shape[0],
                                                                      random_state=1)
    X_train_balanced = np.vstack((X_train_majority_downsampled, X_train_middle, X_train_minority_upsampled))
    y_train_balanced = np.hstack((y_train_majority_downsampled, y_train_middle, y_train_minority_upsampled))

    perm = np.random.permutation(len(X_train_balanced))

    X_train = X_train_balanced[perm]
    y_train = y_train_balanced[perm]

from datetime import datetime
from time import strftime

# timestamp = datetime.now().replace(microsecond=0).isoformat()
timestamp = datetime.now().strftime("%Y-%m-%dT%H:%M:%SZ")



INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Using role: arn:aws:iam::801638386573:role/service-role/AmazonSageMaker-ExecutionRole-20260227T135383


In [6]:
from sagemaker.feature_store.feature_group import FeatureGroup

import time
from time import strftime, gmtime
irrigation_feature_group_name = 'irrigation-feature-group-' + strftime('%d-%H-%M-%S', gmtime())

irrigation_feature_group = FeatureGroup(
    name=irrigation_feature_group_name, sagemaker_session=sm
)

current_time_sec = int(round(time.time()))
record_identifier_feature_name = "id"
df_tv['EventTime'] = pd.Series([current_time_sec]*len(df_tv), dtype="float64")

irrigation_feature_group.load_feature_definitions(data_frame=df_tv)

irrigation_feature_group.create(
    s3_uri=f"s3://{bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name="EventTime",
    role_arn=role,
    enable_online_store=False
)

irrigation_feature_group.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:801638386573:feature-group/irrigation-feature-group-07-14-26-26',
 'FeatureGroupName': 'irrigation-feature-group-07-14-26-26',
 'RecordIdentifierFeatureName': 'id',
 'EventTimeFeatureName': 'EventTime',
 'FeatureDefinitions': [{'FeatureName': 'id', 'FeatureType': 'Integral'},
  {'FeatureName': 'Soil_Type', 'FeatureType': 'String'},
  {'FeatureName': 'Soil_pH', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Soil_Moisture', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Organic_Carbon', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Electrical_Conductivity', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Temperature_C', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Humidity', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Rainfall_mm', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Sunlight_Hours', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Wind_Speed_kmh', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Crop_Type', '

In [7]:
def check_feature_group_status(feature_group):
    status = feature_group.describe().get("FeatureGroupStatus")
    while status == "Creating":
        print("Waiting for Feature Group to be Created")
        time.sleep(5)
        status = feature_group.describe().get("FeatureGroupStatus")
    print(f"FeatureGroup {feature_group.name} successfully created.")

check_feature_group_status(irrigation_feature_group)


Waiting for Feature Group to be Created
Waiting for Feature Group to be Created
Waiting for Feature Group to be Created
FeatureGroup irrigation-feature-group-07-14-26-26 successfully created.


In [ ]:
irrigation_feature_group.ingest(
    data_frame=df_tv, max_workers=20, max_processes=4, 
    wait=True, profile_name='sagemaker'
)   
customer_id = 100
#region = sm.boto_region_name
#sample_record = sm.boto_session.client('sagemaker-featurestore-runtime', region_name=region).get_record(FeatureGroupName=irrigation_feature_group_name, RecordIdentifierValueAsString=str(customer_id))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:7                                                                                    │
│                                                                                                  │
│   4 )                                                                                            │
│   5 customer_id = 100                                                                            │
│   6 region = sm.boto_region_name                                                                 │
│ ❱ 7 sample_record = sm.boto_session.client('sagemaker-featurestore-runtime', region_name=reg     │
│   8                                                                                              │
│                                                                                                  │
│ /home/ryanmcgreevy/Python-venvs/ml/lib/python3.10/site-packages/botocore/client.py:602 in        │
│ _api_call                                                                                        │
│                                                                                                  │
│    599 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    600 │   │   │   │   )                                                                         │
│    601 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  602 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    603 │   │                                                                                     │
│    604 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    605                                                                                           │
│                                                                                                  │
│ /home/ryanmcgreevy/Python-venvs/ml/lib/python3.10/site-packages/botocore/context.py:123 in       │
│ wrapper                                                                                          │
│                                                                                                  │
│   120 │   │   │   with start_as_current_context():                                               │
│   121 │   │   │   │   if hook:                                                                   │
│   122 │   │   │   │   │   hook()                                                                 │
│ ❱ 123 │   │   │   │   return func(*args, **kwargs)                                               │
│   124 │   │                                                                                      │
│   125 │   │   return wrapper                                                                     │
│   126                                                                                            │
│                                                                                                  │
│ /home/ryanmcgreevy/Python-venvs/ml/lib/python3.10/site-packages/botocore/client.py:1078 in       │
│ _make_api_call                                                                                   │
│                                                                                                  │
│   1075 │   │   │   │   'error_code_override'                                                     │
│   1076 │   │   │   ) or error_info.get("Code")                                                   │
│   1077 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1078 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1079 │   │   else:                                                                             │
│   1080 │   │   │   return parsed_response                  

In [ ]:
from sagemaker.feature_store.feature_store import FeatureStore

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)

sagemaker_client = boto_session.client(
    service_name="sagemaker", region_name=region
 )
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime",region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)

feature_store = FeatureStore(feature_store_session) 

base_fg_name = "irrigation-feature-group-07-14-26-26"
base_fg = FeatureGroup(name=base_fg_name, sagemaker_session=feature_store_session)
base_fg.
df_test = feature_store.create_dataset(
    base=base_fg,
    output_path=f"s3://{bucket_name}/{prefix}dataset",

).to_dataframe()[0]
print(df_test.head())

       id Soil_Type  Soil_pH  Soil_Moisture  Organic_Carbon  \
0  203822     Loamy     6.20          45.02            1.35   
1   46601     Loamy     8.00          39.86            0.35   
2  479532     Sandy     6.24          62.95            1.09   
3  479558     Sandy     8.18          45.59            1.55   
4  283059     Sandy     4.98          48.25            0.74   

   Electrical_Conductivity  Temperature_C  Humidity  Rainfall_mm  \
0                     2.70          21.70     67.86      1191.04   
1                     0.27          24.71     61.30      2228.82   
2                     2.07          25.53     57.53       875.72   
3                     2.60          18.39     81.96      1780.74   
4                     2.23          25.06     89.78       814.25   

   Sunlight_Hours  ...  Crop_Growth_Stage  Season Irrigation_Type  \
0            4.67  ...         Vegetative  Kharif            Drip   
1            9.83  ...             Sowing  Kharif         Rainfed   
2    

In [14]:
from sagemaker.feature_store.feature_group import FeatureGroup

import time
from time import strftime, gmtime
irrigation_feature_group_processed_name = 'irrigation-feature-group-processed' + strftime('%d-%H-%M-%S', gmtime())
processed_fg = FeatureGroup(
    name=irrigation_feature_group_processed_name, sagemaker_session=sm
)

current_time_sec = int(round(time.time()))
record_identifier_feature_name = "id"
df_tv['EventTime'] = pd.Series([current_time_sec]*len(df_tv), dtype="float64")

processed_fg.load_feature_definitions(data_frame=df_tv)

processed_fg.create(
    s3_uri=f"s3://{bucket_name}/{prefix}",
    record_identifier_name=record_identifier_feature_name,
    event_time_feature_name="EventTime",
    role_arn=role,
    enable_online_store=False
)

processed_fg.describe()

{'FeatureGroupArn': 'arn:aws:sagemaker:us-east-1:801638386573:feature-group/irrigation-feature-group-processed07-15-58-12',
 'FeatureGroupName': 'irrigation-feature-group-processed07-15-58-12',
 'RecordIdentifierFeatureName': 'id',
 'EventTimeFeatureName': 'EventTime',
 'FeatureDefinitions': [{'FeatureName': 'id', 'FeatureType': 'Integral'},
  {'FeatureName': 'Soil_Type', 'FeatureType': 'String'},
  {'FeatureName': 'Soil_pH', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Soil_Moisture', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Organic_Carbon', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Electrical_Conductivity', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Temperature_C', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Humidity', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Rainfall_mm', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Sunlight_Hours', 'FeatureType': 'Fractional'},
  {'FeatureName': 'Wind_Speed_kmh', 'FeatureType': 'Fractional'},
  {'FeatureNam

In [15]:
check_feature_group_status(processed_fg)

FeatureGroup irrigation-feature-group-processed07-15-58-12 successfully created.


In [28]:
import time
from sagemaker.experiments.experiment import Experiment


#sm = session.client(service_name="sagemaker", region_name=region)

timestamp = int(time.time())

experiment = Experiment.create(
    experiment_name="Irrigation-test-{}".format(timestamp),
    description="Testing irrigation experiment",
    sagemaker_session=sm,
)

experiment_name = experiment.experiment_name
print("Experiment name: {}".format(experiment_name))

Experiment name: Irrigation-test-1775579627


In [40]:


from sagemaker.sklearn.processing import SKLearnProcessor

processing_instance_type = "ml.c5.2xlarge"
processing_instance_count = 1
validation_split_percentage = 0.2
balance_dataset = True


processor = SKLearnProcessor(
    framework_version="1.4-2",
    role=role,
    instance_type=processing_instance_type,
    instance_count=processing_instance_count,
    env={"AWS_DEFAULT_REGION": region},
    max_runtime_in_seconds=7200,
)



INFO:botocore.credentials:Found credentials in environment variables.


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


In [ ]:
from sagemaker.processing import ProcessingInput, ProcessingOutput

processor.run(
    code="preprocess-scikit-text-to-bert-feature-store.py",
    inputs=[
        ProcessingInput(
            input_name="raw-input-data",
            source='s3://{}/{}/train.csv'.format(bucket_name, prefix),
            destination="/opt/ml/processing/input/data/",
            s3_data_distribution_type="ShardedByS3Key",
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="irrigation-train", s3_upload_mode="EndOfJob", source="/opt/ml/processing/output/bert/train"
        ),
        ProcessingOutput(
            output_name="irrigation-validation",
            s3_upload_mode="EndOfJob",
            source="/opt/ml/processing/output/bert/validation",
        ),
        ProcessingOutput(
            output_name="irrigation-test", s3_upload_mode="EndOfJob", source="/opt/ml/processing/output/bert/test"
        ),
    ],
    arguments=[
        "--train-split-percentage",
        str(train_split_percentage),
        "--validation-split-percentage",
        str(validation_split_percentage),
        "--test-split-percentage",
        str(test_split_percentage),
        "--max-seq-length",
        str(max_seq_length),
        "--balance-dataset",
        str(balance_dataset),
        "--feature-store-offline-prefix",
        str(feature_store_offline_prefix),
        "--feature-group-name",
        str(feature_group_name),
    ],
    #experiment_config=experiment_config,
    logs=True,
    wait=False,
)

In [ ]:
# from sagemaker.feature_store.feature_processor import CSVDataSource, feature_processor

# CSV_DATA_SOURCE = CSVDataSource(f's3://{bucket_name}/{prefix}/train.csv')
# OUTPUT_FG = processed_fg.describe()['FeatureGroupArn']

# @feature_processor(inputs=[CSV_DATA_SOURCE], output=OUTPUT_FG)
# def transform(csv_input_df):
#    return csv_input_df
   
# transform()



INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.
your 131072x1 screen size is bogus. expect trouble


26/04/07 12:13:57 WARN Utils: Your hostname, Phobos resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/07 12:13:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/ryanmcgreevy/Python-venvs/ml/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ryanmcgreevy/.ivy2/cache
The jars for the packages stored in: /home/ryanmcgreevy/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
org.apache.hadoop#hadoop-common added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c558e8c1-c650-4c55-89ed-fa238623201b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.1 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.901 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.hadoop#hadoop-common;3.3.1 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-protobuf_3_7;1.1.1 in central
	found org.apache.hadoop#hadoop-annotations;3.3.1 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.1.1 in central
	found com.google.guava#guava;27.0-jre in central
	found com.google.guava#failureaccess;1.0 in central
	found com.google.guava#listenablefuture;9999.0-empty-to-avoid-conflict-with-guava in central

26/04/07 12:14:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
INFO:sagemaker:Loading data from s3a://jrm-kaggle/playgrounds6ep4//train.csv.


26/04/07 12:15:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/04/07 12:15:10 WARN BasicProfileConfigLoader: Your profile name includes a 'profile ' prefix. This is considered part of the profile name in the Java SDK, so you will need to include this prefix in your profile name when you reference this profile from your Java code.
26/04/07 12:15:10 WARN BasicProfileConfigLoader: Your profile name includes a 'profile ' prefix. This is considered part of the profile name in the Java SDK, so you will need to include this prefix in your profile name when you reference this profile from your Java code.
26/04/07 12:15:10 WARN BasicProfileConfigLoader: Your profile name includes a 'profile ' prefix. This is considered part of the profile name in the Java SDK, so you will need to include this prefix in your profile name when you reference this profile from your Java code.


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:74                                                                                   │
│                                                                                                  │
│   71 │     y_train = y_train_balanced[perm]                                                      │
│   72    return csv_input_df                                                                      │
│   73                                                                                             │
│ ❱ 74 transform()                                                                                 │
│   75                                                                                             │
│   76                                                                                             │
│                                                                                                  │
│ /home/ryanmcgreevy/Python-venvs/ml/lib/python3.10/site-packages/sagemaker/feature_store/feature_ │
│ processor/_udf_wrapper.py:60 in wrapper                                                          │
│                                                                                                  │
│   57 │   │   │   │   fp_config=fp_config,                                                        │
│   58 │   │   │   )                                                                               │
│   59 │   │   │                                                                                   │
│ ❱ 60 │   │   │   output = udf(*udf_args, **udf_kwargs)                                           │
│   61 │   │   │                                                                                   │
│   62 │   │   │   self.udf_output_receiver.ingest_udf_output(output, fp_config)                   │
│   63                                                                                             │
│                                                                                                  │
│ in transform:13                                                                                  │
│                                                                                                  │
│   10    df_tv = csv_input_df                                                                     │
│   11    #print(df_tv.head())                                                                     │
│   12                                                                                             │
│ ❱ 13    df_x = df_tv.iloc[:,1:-1]                                                                │
│   14                                                                                             │
│   15    df_dummy = pd.get_dummies(df_x, dtype=int, drop_first=False)                             │
│   16    continous_variables = df_dummy.select_dtypes(['float64']).columns                        │
│                                                                                                  │
│ /home/ryanmcgreevy/Python-venvs/ml/lib/python3.10/site-packages/pyspark/sql/dataframe.py:1988 in │
│ __getattr__                                                                                      │
│                                                                                                  │
│   1985 │   │   [Row(age=2), Row(age=5)]                                                          │
│   1986 │   │   """                                                                               │
│   1987 │   │   if name not in self.columns:                                                      │
│ ❱ 1988 │   │   │   raise AttributeError(                                                         │
│   1989 │   │   │   │   "'%s' object has no attribute '%s'" % (self.__class__.__name__, name)     │
│   1990 │   │   │   )                                       